# Notebook 02: Pemrosesan Dataset Wajah Aktor, Pelatihan FaceNet Triplet Loss, & Evaluasi Matrix Score

Notebook ini menjalankan **Fokus 2**, yaitu memproses dataset citra wajah aktor dengan memanggil hasil akhir model detektor wajah `storage/app/ai/weights/hasil_training_yolov8_wajah.pt` (dari **Notebook 01**), kemudian:
1. **Tahap 01:** Memotong kotak wajah aktor & orang random dengan margin `12%`, melakukan augmentasi pose/cahaya, serta menormalisasi piksel ke ukuran **`160×160` RGB**.
2. **Tahap 02:** Melatih arsitektur **`JaringanFaceNet` (`torch.nn.Module`)** dari **Epoch 0** menggunakan fungsi **Triplet Margin Loss (`Anchor, Positive, Negative`)** di **GPU NVIDIA GeForce RTX 4060** hingga menghasilkan `storage/app/ai/weights/hasil_training_facenet_aktor.pth`.
3. **Tahap 03:** Mengekstrak vektor referensi **`512-D` (`.npy`)** ke folder `python/embeddings/`.
4. **Tahap 04:** Menghitung **Euclidean Distance**, mengkalibrasi *threshold* optimal, dan menghasilkan **Confusion Matrix (`TP, TN, FP, FN`)**, **Accuracy**, **Precision**, **Recall**, serta **F1-Score**.

In [2]:
# Mengatur path absolut root proyek agar seluruh modul terbaca dengan benar dari Jupyter Notebook lokal
import sys
from pathlib import Path

FOLDER_NOTEBOOK_INI = Path.cwd().resolve()
if FOLDER_NOTEBOOK_INI.name == "02_training_dataset_wajah_aktor":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent.parent
elif FOLDER_NOTEBOOK_INI.name == "training_python":
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI.parent
else:
    FOLDER_ROOT_PROYEK = FOLDER_NOTEBOOK_INI

for path_modul in [
    FOLDER_ROOT_PROYEK,
    FOLDER_ROOT_PROYEK / "training_python",
    FOLDER_ROOT_PROYEK / "training_python" / "02_training_dataset_wajah_aktor",
    FOLDER_ROOT_PROYEK / "python",
]:
    if str(path_modul) not in sys.path:
        sys.path.insert(0, str(path_modul))

import konfigurasi_training as konfig
from facenet.arsitektur_facenet_manual import JaringanFaceNet
from tahap_01_crop_dan_preprocess_wajah import ekstrak_dan_preprocess_wajah_dataset
from tahap_02_training_facenet_triplet import latih_model_facenet_triplet_loss
from tahap_03_ekstraksi_embedding_aktor import buat_database_embedding_referensi
from tahap_04_evaluasi_euclidean_dan_f1 import kalibrasi_threshold_dan_evaluasi_matrix_f1

# Mengaktifkan mode output unlimited dan memverifikasi penguncian GPU NVIDIA GeForce RTX 4060 Laptop GPU serta bobot YOLOv8 dari Notebook 01
info_output_notebook = konfig.aktifkan_output_notebook_unlimited()
indeks_cuda, objek_device_gpu, info_perangkat_gpu = konfig.pilih_perangkat_gpu_rtx_wajib(wajib_cuda=True)
jaringan_uji = JaringanFaceNet(dimensi_embedding=konfig.DIMENSI_VEKTOR_EMBEDDING_FACENET)

print("Root Proyek             :", FOLDER_ROOT_PROYEK)
print("Perangkat GPU Aktif     :", info_perangkat_gpu["nama_gpu"], f"({info_perangkat_gpu['vram_gb']} GB VRAM)")
print("Indeks CUDA             :", info_perangkat_gpu["device_pytorch"], "-", info_perangkat_gpu["catatan_task_manager"])
print("Bobot YOLOv8 Tersedia   :", konfig.FILE_BOBOT_HASIL_TRAINING_YOLO.exists(), "->", konfig.FILE_BOBOT_HASIL_TRAINING_YOLO)
print("Jumlah Parameter FaceNet:", sum(p.numel() for p in jaringan_uji.parameters()))
print("Output Unlimited        :", info_output_notebook)

Root Proyek             : C:\laragon\www\IndoLens
Perangkat GPU Aktif     : NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB VRAM)
Indeks CUDA             : cuda:0 - Tampil sebagai GPU 1 di Windows Task Manager (GPU 0 adalah Intel UHD Graphics)
Bobot YOLOv8 Tersedia   : True -> C:\laragon\www\IndoLens\storage\app\ai\weights\hasil_training_yolov8_wajah.pt
Jumlah Parameter FaceNet: 5751904
Output Unlimited        : {'berjalan_di_notebook': True, 'jumlah_stream_diatur': 2, 'interval_flush_detik': 0.5, 'interval_progress_detik': 1.5, 'ultralytics_tqdm_diatur': True, 'tqdm_standar_diatur': True, 'konfig_server_jupyter_unlimited': True}


## Tahap 01: Pemotongan Kotak Wajah Aktor (Margin 12%) & Preprocessing `160×160` RGB
Memanggil `hasil_training_yolov8_wajah.pt` untuk mendeteksi wajah pada foto aktor (dari database `/camp` maupun `wajah_aktor_target/`) dan `wajah_orang_random/`, menambahkan margin tepi `12%`, menyekalakan ke `160×160` RGB, membuat variasi augmentasi pose/pencahayaan, serta menyimpan tensor ternormalisasi ke `crop_wajah_160x160/`.

In [4]:
# Mengaktifkan mode output unlimited lalu menjalankan deteksi crop wajah dan preprocessing 160x160 RGB beserta augmentasi data latih
konfig.aktifkan_output_notebook_unlimited()
ringkasan_preprocessing = ekstrak_dan_preprocess_wajah_dataset(
    aktifkan_augmentasi_aktor=True,
    rasio_margin_crop=konfig.RASIO_MARGIN_CROP_WAJAH,
)

  TAHAP 01: DETEKSI CROP WAJAH (YOLOV8) & PREPROCESSING 160x160 RGB
[*] Perangkat Komputasi GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
[*] Bobot Detektor YOLOv8       : C:\laragon\www\IndoLens\storage\app\ai\weights\hasil_training_yolov8_wajah.pt
[*] Resolusi Target FaceNet     : 160x160 piksel (Margin Crop = 4%)

  TAHAP 03: PREPROCESSING & CROP WAJAH 160x160 SELESAI
[OK] Jumlah aktor terproses           : 12 aktor
[OK] Crop wajah aktor asli            : 422 dari 423 foto
[OK] Variasi augmentasi wajah aktor   : 2110 tensor
[OK] Sampel wajah orang random        : 650 tensor
[OK] Manifest disimpan ke             : C:\laragon\www\IndoLens\python\datasets\dataset_facenet_aktor\crop_wajah_160x160\manifest_preprocessing_wajah.json


## Tahap 02: Pelatihan `JaringanFaceNet` dengan Triplet Margin Loss (Epoch 0)
Menyusun pasangan `(Anchor, Positive, Negative)` dan melatih jaringan saraf FaceNet dari Epoch 0 di GPU RTX 4060 hingga konvergen (dilengkapi *Early Stopping*), lalu menyimpan bobot terbaik ke `storage/app/ai/weights/hasil_training_facenet_aktor.pth`.

In [6]:
# Mengaktifkan mode output unlimited lalu melatih JaringanFaceNet menggunakan fungsi Triplet Margin Loss di GPU RTX 4060
konfig.aktifkan_output_notebook_unlimited()
data_hasil_training_facenet = latih_model_facenet_triplet_loss(
    jumlah_epoch=konfig.JUMLAH_EPOCH_TRAINING_FACENET,
    batas_patience=konfig.BATAS_PATIENCE_EARLY_STOPPING_FACENET,
    ukuran_batch=konfig.UKURAN_BATCH_TRIPLET_FACENET,
    laju_belajar_awal=konfig.LAJU_BELAJAR_AWAL_FACENET,
    batas_margin_triplet=konfig.BATAS_MARGIN_TRIPLET_LOSS,
)

  TAHAP 02: PELATIHAN FACENET LANDMARK 3x3 + GDCONV 5x5 & OPEN-SET TRIPLET (RTX 4060)
[*] Perangkat Komputasi GPU     : NVIDIA GeForce RTX 4060 Laptop GPU (8.0 GB VRAM)
[*] Indeks Perangkat            : cuda:0 (Tampil sebagai GPU 1 di Windows Task Manager (GPU 0 adalah Intel UHD Graphics))
[*] Jumlah Aktor & Orang Random : 12 aktor | 650 individu orang random
[*] Target Epoch & Patience     : Maks 60 Epoch (Early Stopping Patience = 12)
[*] Komposisi Batch PK + Random : 12x3 aktor + 14x2 instance random + 8 hard-neg
[*] Dimensi Vektor & Margin     : 512-D | Margin Aktor=0.5, Unknown=0.75

      Epoch    GPU_mem  TrainLoss   TrainAcc   Triplets       Size
       1/60      3.08G     1.7007      99.8%         64        160: 100% ━━━━━━━━━━━━ 40/40 1.2it/s 33.6s1.5ss
                 Split   Triplets    ValLoss    Dist(+)    Dist(-)     ValAcc: 100% ━━━━━━━━━━━━ 10/10 1.7it/s 5.8s.0ss
                   val        640     0.9849      0.787      0.897     100.0%

      Epoch    GPU_mem  Tra

## Tahap 03: Pembuatan Database Reference Embedding `512-D` (`.npy`)
Mengekstrak vektor embedding `512-D` (`L2-normalized`) untuk setiap foto asli aktor memakai `hasil_training_facenet_aktor.pth` dan menyimpannya ke `python/embeddings/{slug_aktor}/{image_id}.npy` beserta `manifest.json`.

In [8]:
# Mengaktifkan mode output unlimited lalu mengekstrak dan menyimpan vektor reference embedding .npy 512-D ke folder python/embeddings/
konfig.aktifkan_output_notebook_unlimited()
ringkasan_ekstraksi_embedding = buat_database_embedding_referensi()

  TAHAP 03: EKSTRAKSI DATABASE EMBEDDING REFERENSI 512-D (FACENET)
[*] Perangkat Komputasi GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
[*] Bobot FaceNet Digunakan     : C:\laragon\www\IndoLens\storage\app\ai\weights\hasil_training_facenet_aktor.pth
[*] Folder Tujuan Embedding     : C:\laragon\www\IndoLens\python\embeddings
[OK] Total aktor memiliki embedding : 12 aktor
[OK] Total file .npy 512-D disimpan : 422 file
[OK] File manifest.json ditulis ke  : C:\laragon\www\IndoLens\python\embeddings\manifest.json
[*] Status sinkronisasi runtime     : Layanan yolo_service (port 8002) sedang tidak dijalankan (vektor .npy di disk akan dimuat otomatis saat web dinyalakan).


## Tahap 04: Evaluasi Euclidean Distance, Kalibrasi Threshold, Confusion Matrix, & F1-Score
Menghitung distribusi jarak Euclidean *intra-class* vs *inter-class*, mencari nilai `batas_ambang_jarak` (*threshold*) optimal, serta menghasilkan tabel **Confusion Matrix (`TP, TN, FP, FN`)** dan metrik **Accuracy, Precision, Recall, serta F1-Score**.

In [10]:
# Mengaktifkan mode output unlimited lalu menjalankan kalibrasi threshold Euclidean Distance dan evaluasi Confusion Matrix serta F1-Score
konfig.aktifkan_output_notebook_unlimited()
laporan_evaluasi_lengkap = kalibrasi_threshold_dan_evaluasi_matrix_f1()

  TAHAP 04: EVALUASI EUCLIDEAN DISTANCE, CONFUSION MATRIX, & F1-SCORE
[*] Perangkat Komputasi GPU     : NVIDIA GeForce RTX 4060 Laptop GPU
[*] Jumlah Aktor Referensi      : 12 aktor
[OK] Batas Ambang Jarak Optimal (Threshold) : 0.55
[OK] Accuracy                               : 98.41%
[OK] Precision                              : 99.51%
[OK] Recall                                 : 96.43%
[OK] F1-Score                               : 97.94%
[OK] Penolakan Unknown (Specificity / TNR)  : 99.69%
[OK] Dimensi Efektif 95% (SVD)              : 11 dimensi (Top-3 var = 36.1%)
[OK] Komponen Confusion Matrix              : TP=405, TN=650, FP=2, FN=15
[OK] Gambar Confusion Matrix disimpan ke    : C:\laragon\www\IndoLens\training_python\artefak_hasil_training\04_hasil_evaluasi_dan_matrix_score\confusion_matrix_pengenalan_aktor.png
[OK] Laporan JSON evaluasi disimpan ke      : C:\laragon\www\IndoLens\training_python\artefak_hasil_training\04_hasil_evaluasi_dan_matrix_score\laporan_evaluasi_matrix_d